# RIDGE + XGBOOST ENSEMBLE
Blend Ridge Regression (good generalization) with XGBoost (signal capture)

**Insight:** Ridge CV R² = 0.0129 (stable, generalizes well), but XGBoost overfits badly.

**Strategy:**
1. Train Ridge Regression (alpha=10.0, proven stable)
2. Train XGBoost on 150 best features
3. Blend predictions: 30% Ridge + 70% XGBoost
   - Ridge dampens overfitting
   - XGBoost captures signal
4. Optimize blend ratio if needed

**Expected outcome:** Better CV-Test generalization

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
import gc
import warnings
warnings.filterwarnings('ignore')

print("="*70)
print("RIDGE + XGBOOST ENSEMBLE")
print("="*70)

# ============== LOAD DATA ==============
print("\n[1/5] Loading data...")
train_df = pd.read_parquet('/kaggle/input/competitions/short-horizon-return-prediction-challenge-by-i-rage/train.parquet')
test_df = pd.read_parquet('/kaggle/input/competitions/short-horizon-return-prediction-challenge-by-i-rage/test.parquet')

def compress_dtypes(df):
    for col in df.columns:
        if df[col].dtype != 'object':
            c_min, c_max = df[col].min(), df[col].max()
            if str(df[col].dtype)[:3] == 'int':
                if c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
            else:
                if c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
    return df

train_df = compress_dtypes(train_df)
test_df = compress_dtypes(test_df)
train_df = train_df.fillna(train_df.median())
test_df = test_df.fillna(test_df.median())

y_train = train_df['TARGET'].values.copy()
test_ids = test_df['ID'].values.copy()

X_train = train_df.drop(['ID', 'TARGET'], axis=1)
X_test = test_df.drop(['ID'], axis=1)

const_cols = list(set(X_train.columns[X_train.nunique() <= 1]) | set(X_test.columns[X_test.nunique() <= 1]))
X_train = X_train.drop(const_cols, axis=1, errors='ignore')
X_test = X_test.drop(const_cols, axis=1, errors='ignore')

common_cols = sorted(list(set(X_train.columns) & set(X_test.columns)))
X_train = X_train[common_cols].copy()
X_test = X_test[common_cols].copy()

del train_df, test_df
gc.collect()
print(f"✓ Train: {X_train.shape}, Test: {X_test.shape}")

# ============== EXTRACT TOP 150 FEATURES ==============
print("\n[2/5] Extracting top 150 features by importance...")

temp_model = xgb.XGBRegressor(
    objective='reg:squarederror',
    max_depth=5, learning_rate=0.05,
    n_estimators=100, random_state=42, verbosity=0
)
temp_model.fit(X_train, y_train)
importance_dict = temp_model.get_booster().get_score(importance_type='weight')

importance_df = pd.DataFrame([
    {'feature': k, 'importance': v}
    for k, v in importance_dict.items()
]).sort_values('importance', ascending=False)

top_150_features = importance_df.head(150)['feature'].tolist()
X_train_feat = X_train[top_150_features].copy()
X_test_feat = X_test[top_150_features].copy()
print(f"✓ Using top 150 features")

del temp_model
gc.collect()

In [ ]:
# ============== TRAIN RIDGE REGRESSION ==============
print("\n[3/5] Training Ridge Regression (alpha=10.0)...")

kf = KFold(n_splits=5, shuffle=True, random_state=42)

ridge_cv_preds = np.zeros(len(X_train_feat))
ridge_test_preds = np.zeros(len(X_test_feat))
ridge_cv_scores = []

for fold_num, (train_idx, val_idx) in enumerate(kf.split(X_train_feat), 1):
    X_tr, X_val = X_train_feat.iloc[train_idx], X_train_feat.iloc[val_idx]
    y_tr, y_val = y_train[train_idx], y_train[val_idx]
    
    # Standardize for Ridge
    scaler = StandardScaler()
    X_tr_scaled = scaler.fit_transform(X_tr)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test_feat)
    
    ridge = Ridge(alpha=10.0)
    ridge.fit(X_tr_scaled, y_tr)
    
    y_pred_val = ridge.predict(X_val_scaled)
    r2 = r2_score(y_val, y_pred_val)
    ridge_cv_scores.append(r2)
    ridge_cv_preds[val_idx] = y_pred_val
    ridge_test_preds += ridge.predict(X_test_scaled) / 5
    
    print(f"  Fold {fold_num} R²: {r2:.6f}")
    del ridge, scaler, X_tr_scaled, X_val_scaled, X_test_scaled
    gc.collect()

ridge_avg_r2 = np.mean(ridge_cv_scores)
print(f"\n✓ Ridge Avg R²: {ridge_avg_r2:.6f}")

In [ ]:
# ============== TRAIN XGBOOST ==============
print("\n[3/5] Training XGBoost (max_depth=5, 150 features)...")

xgb_cv_preds = np.zeros(len(X_train_feat))
xgb_test_preds = np.zeros(len(X_test_feat))
xgb_cv_scores = []

xgb_params = {
    'objective': 'reg:squarederror',
    'max_depth': 5,
    'learning_rate': 0.05,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'lambda': 2.0,
    'alpha': 1.0,
    'random_state': 42,
    'verbosity': 0
}

for fold_num, (train_idx, val_idx) in enumerate(kf.split(X_train_feat), 1):
    X_tr, X_val = X_train_feat.iloc[train_idx], X_train_feat.iloc[val_idx]
    y_tr, y_val = y_train[train_idx], y_train[val_idx]
    
    model = xgb.XGBRegressor(**xgb_params, n_estimators=400)
    model.fit(X_tr, y_tr)
    
    y_pred_val = model.predict(X_val)
    r2 = r2_score(y_val, y_pred_val)
    xgb_cv_scores.append(r2)
    xgb_cv_preds[val_idx] = y_pred_val
    xgb_test_preds += model.predict(X_test_feat) / 5
    
    print(f"  Fold {fold_num} R²: {r2:.6f}")
    del model, X_tr, X_val
    gc.collect()

xgb_avg_r2 = np.mean(xgb_cv_scores)
print(f"\n✓ XGBoost Avg R²: {xgb_avg_r2:.6f}")

In [ ]:
# ============== ENSEMBLE BLENDING ==============
print("\n[4/5] Testing ensemble blend ratios...\n")

blend_ratios = [
    (1.0, 0.0),   # Pure Ridge
    (0.5, 0.5),   # 50-50 blend
    (0.3, 0.7),   # 30% Ridge, 70% XGBoost (recommended)
    (0.2, 0.8),   # 20% Ridge, 80% XGBoost
    (0.0, 1.0)    # Pure XGBoost
]

blend_results = []

for ridge_w, xgb_w in blend_ratios:
    # Blend validation predictions
    blended_cv_preds = ridge_w * ridge_cv_preds + xgb_w * xgb_cv_preds
    r2_cv = r2_score(y_train, blended_cv_preds)
    
    # Blend test predictions  
    blended_test_preds = ridge_w * ridge_test_preds + xgb_w * xgb_test_preds
    
    blend_results.append({
        'Ridge Weight': ridge_w,
        'XGBoost Weight': xgb_w,
        'CV R²': r2_cv,
        'Predictions': blended_test_preds
    })
    
    print(f"Ridge {ridge_w:.1f} + XGBoost {xgb_w:.1f}: CV R² = {r2_cv:.6f}")

blend_df = pd.DataFrame([
    {'Ridge %': f"{r['Ridge Weight']*100:.0f}%",
     'XGBoost %': f"{r['XGBoost Weight']*100:.0f}%",
     'CV R²': r['CV R²']}
    for r in blend_results
])

best_blend_idx = np.argmax([r['CV R²'] for r in blend_results])
best_blend = blend_results[best_blend_idx]
best_r2 = best_blend['CV R²']
best_preds = best_blend['Predictions']

print("\n" + "="*70)
print("BLEND COMPARISON")
print("="*70)
print(blend_df.to_string(index=False))
print("\n" + "="*70)
print(f"🎯 BEST BLEND: Ridge {blend_results[best_blend_idx]['Ridge Weight']:.1f} + XGBoost {blend_results[best_blend_idx]['XGBoost Weight']:.1f}")
print(f"CV R² = {best_r2:.6f}")
print("="*70)

In [ ]:
# ============== COMPARISON WITH BASELINES ==============
print("\n" + "="*70)
print("ANALYSIS: Single Models vs Ensemble")
print("="*70)

comparison = pd.DataFrame([
    {'Model': 'Ridge Alone', 'CV R²': ridge_avg_r2, 'Type': 'Linear'},
    {'Model': 'XGBoost Alone', 'CV R²': xgb_avg_r2, 'Type': 'Tree'},
    {'Model': 'Best Ensemble', 'CV R²': best_r2, 'Type': 'Blend'}
]).sort_values('CV R²', ascending=False)

print(comparison.to_string(index=False))

print(f"\n" + "="*70)
print("RESULTS vs CURRENT BEST:")
print("="*70)

current_best = -0.01059  # From solution_target_transform (test)
best_cv_prev = 0.1356      # From solution_feature_selection (CV)

print(f"Previous CV R² (150-feature XGBoost): {best_cv_prev:.6f}")
print(f"Previous Test R²:                     {current_best:.6f}")
print(f"\nNew Ensemble CV R²:                   {best_r2:.6f}")

if best_r2 > best_cv_prev:
    improvement_cv = ((best_r2 - best_cv_prev) / abs(best_cv_prev)) * 100
    print(f"\n✅ CV R² IMPROVED by {improvement_cv:.2f}%")
    print(f"   Ensemble's CV is better → should generalize better to test!")
elif best_r2 > 0.05:
    print(f"\n⭐ GOOD! Ensemble CV R² = {best_r2:.6f} (vs Ridge baseline 0.0129)")
    print(f"   Much better signal than Ridge alone")
else:
    print(f"\n⚠️  Ensemble CV R² = {best_r2:.6f} not better than pure Ridge ({ridge_avg_r2:.6f})")
    print(f"    Likely pure XGBoost (-0.01059) still best for test")

print("="*70)

In [ ]:
# ============== CREATE SUBMISSION ==============
print("\n[5/5] Creating submission...")

submission = pd.DataFrame({'ID': test_ids, 'TARGET': best_preds})
print(f"✓ Submission shape: {submission.shape}")
print(f"✓ Target - Mean: {submission['TARGET'].mean():.6f}, Std: {submission['TARGET'].std():.6f}")
print(f"✓ Target - Min: {submission['TARGET'].min():.6f}, Max: {submission['TARGET'].max():.6f}")

ridge_w_best = blend_results[best_blend_idx]['Ridge Weight']
xgb_w_best = blend_results[best_blend_idx]['XGBoost Weight']
submission_file = f'submission_ensemble_ridge{ridge_w_best:.1f}_xgb{xgb_w_best:.1f}.csv'
submission.to_csv(submission_file, index=False)
print(f"✓ Saved: {submission_file}")

print("\n" + "="*70)
print("FINAL SUMMARY")
print("="*70)
print(f"\nEnsemble Composition:")
print(f"  └─ {ridge_w_best*100:.0f}% Ridge (alpha=10.0)")
print(f"  └─ {xgb_w_best*100:.0f}% XGBoost (150 features)")
print(f"\nPerformance:")
print(f"  Ensemble CV R²: {best_r2:.6f}")
print(f"  Ridge CV R²:    {ridge_avg_r2:.6f} (stable baseline)")
print(f"  XGBoost CV R²:  {xgb_avg_r2:.6f} (signal capture)")
print(f"\nSubmission File: {submission_file}")
print(f"\nNote: This balances Ridge's generalization with XGBoost's signal detection.")
print(f"      Expect modest improvement over pure XGBoost's test R² = -0.01059")
print("="*70)